# Exercise 2 - Training a CNN on CIFAR-10

Work in one shared notebook with your group. Some code is provided and some cells contain only `# Write code here`.

## 0. Setup

The following cell imports the required packages. You are not expected to train any model on your gpu, using your cpu suffices.

You might have to use terminal commands such as `pip install tensorflow keras`. For mac use `pip3`, and if you have different python versions installed consider `py -3.<version> -m pip install <packagename>`.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import sklearn
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.datasets import fetch_openml
from scikeras.wrappers import KerasClassifier

## 1. Data collection

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) contains 60,000 colour images from 10 classes. Each image is 32 by 32 pixels. The following cell downloads the data and loads the 50,000 training images and 10,000 test images. The class names are stored in label order.
The smaller dataset contains a 20,000 instance stratified sample of the original CIFAR-10 dataset.

In [4]:
smaller_dataset = True

if smaller_dataset:
    print("Downloading ~15MB...")
    cifar_small = fetch_openml('CIFAR_10_small', version=1, as_frame=False, parser='auto')

    X = cifar_small.data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
    y = cifar_small.target.astype(np.int64)

    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
else: 
    print("Downloading ~170MB...")
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print("Dataset downloaded.")

y_train = y_train.ravel()
y_test = y_test.ravel()

class_names = np.array([
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
])

Dataset downloaded.


The labels are flattened from `(n, 1)` to `(n,)` because the later scikit-learn functions expect a one-dimensional target array. `class_names` lets us replace numeric labels with readable names.

Print the shapes of `x_train`, `y_train`, `x_test`, and `y_test`. Then display the first 10 training images in a 2 by 5 figure. Place the correct class name above each image and remove the axes.

In [ ]:
# Write code here

This check confirms that the image dimensions and labels match what we expect before training.

An individual image has shape `(32, 32, 3)`. Explain what the three dimensions represent. Then identify the input and output of the classifier.

**Answer:**

## 2. Cleaning and preprocessing

CIFAR-10 is already labelled and does not require missing-value treatment. Its pixels are integers from 0 to 255. Convert both image arrays to `float32` and divide them by 255 so that every pixel is between 0 and 1. Store the results using the same variable names, then print the data type, minimum, and maximum of `x_train` to check the result.

In [ ]:
# Write code here

Scaling changes each RGB channel from 0-255 to 0-1 without removing colour. Smaller input values usually make gradient-based training more stable.

## 3. Feature engineering

No features will be created manually. During training, the convolutional layers learn filters that respond to useful local patterns such as edges, shapes, and textures.

## 4. Train, validation, and test data

The 10,000 test images will remain untouched while models are compared. `StratifiedKFold` creates five train-validation splits from the 50,000 training images while preserving the class proportions. Each image is used for validation once.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2112)

## 5. CNN structure

`create_cnn()` returns a new, untrained model. Convolutional layers learn filters, pooling reduces the size of the feature maps, and dense layers perform the final classification. The output contains probabilities for the 10 classes. Cross-entropy measures prediction error, and Adam updates the weights using gradients calculated through backpropagation.

In [ ]:
def create_cnn(conv1_filters=32, conv2_filters=64, dense_units=64):
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(conv1_filters, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(conv2_filters, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

Keeping the architecture in a function makes it easy to create new versions by changing only a few settings.

The following cell creates the standard CNN and displays its layers, output shapes, and parameter counts. It does not train the model.

In [ ]:
standard_model = create_cnn()
standard_model.summary()

Checking the summary now helps catch shape errors before training, when they are quicker to fix.

## 6. Cross-validation and model selection

Complete the next cell by defining `cross_validate_cnn(conv1_filters=32, conv2_filters=64, dense_units=64, n_folds=5, epochs=3)`.

The function should set up stratified k-fold cross-validation using `n_folds`. For each fold, create and train a fresh CNN using the specified model settings. Use `batch_size=128` and `verbose=0`. Evaluate each model using that fold's validation data. Do not use the test data during cross-validation.

You may use scikit-learn's `cross_validate()` or implement the folds directly.

The function should return:

- The validation accuracy from each fold
- The mean validation accuracy
- The standard deviation of the validation accuracies

With the default settings, the function should return five fold accuracies, their mean, and their standard deviation.

Call the function with its default settings and store the returned values as `standard_model_scores`, `standard_model_mean`, and `standard_model_std`.

In [ ]:
# Write code here
# def cross_validate_cnn(conv1_filters=32, conv2_filters=64, dense_units=64, epochs=3):




# standard_scores, standard_mean, standard_std = cross_validate_cnn()

Try some combinations of different `conv1_filters`, `conv2_filters`, and `dense_units`. Call `cross_validate_cnn()` with the changed values and store the result as `modified_model_scores`. Keep the test data untouched.

In [ ]:
# Write code here


# modified_model_scores, modified_model_mean, modified_model_std = cross_validate_cnn(conv1_filters=.., conv2_filters=..., dense_units=...)

Record the results before selecting a model.

| Configuration | Conv 1 filters | Conv 2 filters | Dense units | Fold accuracies | Mean accuracy |
|---|---:|---:|---:|---|---:|
| Standard | 32 | 64 | 64 |  |  |
| Modified |  |  |  |  |  |

Which configuration will you use, and what evidence supports this choice?

**Answer:**

## 7. Final training and test evaluation

Complete the next cell using your selected settings. Create a fresh model named `final_model`, train it on all of `x_train` and `y_train` for three epochs with `batch_size=128`, and generate predictions for `x_test`. Convert the predicted probabilities to class labels using `np.argmax(..., axis=1)` and store them as `y_pred`. Print the test accuracy using `accuracy_score(y_test, y_pred)`.

This is the only time the test data should be used. Do not change the model after viewing the result.

In [ ]:
# Write code here

Training the chosen model on all training images uses all available data for learning. The test result then measures performance on images that were not used to choose the model settings.

The following cell displays a normalised confusion matrix. Each row represents a correct class, so larger off-diagonal values show which classes the CNN confuses.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    normalize="true",
    values_format=".2f",
    cmap="Blues",
    xticks_rotation=45,
    colorbar=False,
    ax=ax
)
ax.set_title("Normalised test confusion matrix")
plt.tight_layout()
plt.show()

Normalising each row turns the counts into proportions, which makes the classes easier to compare.

## 8. Discussion

1. Why must every cross-validation fold begin with a fresh CNN?
2. Why was the test set kept separate while the model settings were changed?
3. How are backpropagation and gradient-based optimisation used during `model.fit()`?
4. Is ~55% test accuracy significantly better than a random classifier?
4. Which classes were most frequently confused? Suggest one visual reason.
5. Did the modified model improve consistently across the five folds, or only in one fold?